# Multi-Material Multi-ZAID Perturbation Test — Fe-56 & Fe-54 in PWR Sphere

Test the multi-material PERT card generation and perturbed material creation for **two ZAIDs** across **two materials** with **two reactions**:

| ZAID | Material | Pert Mat ID | Cells | Description |
|------|----------|-------------|-------|-------------|
| 26056 (Fe-56) | 400000 | 41 | 3, 5, 7 | Structural Steel |
| 26056 (Fe-56) | 300000 | 31 | 9 | Vessel Steel |
| 26054 (Fe-54) | 400000 | 42 | 3, 5, 7 | Structural Steel |
| 26054 (Fe-54) | 300000 | 32 | 9 | Vessel Steel |

- **Reactions**: MT=2 (elastic), MT=51 (discrete inelastic level 1)
- **Order**: 1 (first-order only)
- **Energy grid**: SCALE56 (56 bins)
- **Expected PERT cards**: 2 ZAIDs × 2 materials × 2 reactions × 56 bins = **448**

In [1]:
import shutil
from pathlib import Path
import kika
from kika.energy_grids import SCALE56

# Work on a copy to avoid modifying the original
original = Path("/share_snc/snc/JuanMonleon/PWRSphere/PERT/sum/PWRSphere_Fe56.i")
workdir = Path("./test_output")
workdir.mkdir(exist_ok=True)
workfile = workdir / original.name
shutil.copy(original, workfile)

print(f"Working copy: {workfile}")
print(f"SCALE56 energy grid: {len(SCALE56)-1} bins")

Working copy: test_output/PWRSphere_Fe56.i
SCALE56 energy grid: 56 bins


## 1. Read original input and inspect materials

In [2]:
inp = kika.read_mcnp(str(workfile))
print(inp)

                      MCNP Input Data                       

------------------------------------------------------------
                       PERTURBATIONS                        
------------------------------------------------------------
Number of perturbations:  0
Number of perturbations:  0 (empty collection)

Use .perturbation to access perturbation data.



## 2. Create perturbed materials for both ZAIDs

In [3]:
# Perturbed materials for ZAID 26056 (Fe-56)
pert_m40_56, pert_m30_56 = kika.perturb_materials(
    inp.materials,
    material_ids=[400000, 300000],
    nuclide=26056,
    density=[-7.85, -7.85],
    pert_mat_id=[41, 31],
)
print(f"Perturbed material 41 (26056 from m400000): density={pert_m40_56.density:.4f} g/cc")
print(f"Perturbed material 31 (26056 from m300000): density={pert_m30_56.density:.4f} g/cc")

Density change: 7.8500e+00 → 1.4828e+01 g/cm³
Perturbed material 41 added to collection (100% increase in 26056)
Density change: 7.8500e+00 → 1.4828e+01 g/cm³
Perturbed material 31 added to collection (100% increase in 26056)
Perturbed material 41 (26056 from m400000): density=14.8278 g/cc
Perturbed material 31 (26056 from m300000): density=14.8278 g/cc


In [4]:
# Perturbed materials for ZAID 26054 (Fe-54)
pert_m40_54, pert_m30_54 = kika.perturb_materials(
    inp.materials,
    material_ids=[400000, 300000],
    nuclide=26054,
    density=[-7.85, -7.85],
    pert_mat_id=[42, 32],
)
print(f"Perturbed material 42 (26054 from m400000): density={pert_m40_54.density:.4f} g/cc")
print(f"Perturbed material 32 (26054 from m300000): density={pert_m30_54.density:.4f} g/cc")

Density change: 7.8500e+00 → 8.2751e+00 g/cm³
Perturbed material 42 added to collection (100% increase in 26054)
Density change: 7.8500e+00 → 8.2751e+00 g/cm³
Perturbed material 32 added to collection (100% increase in 26054)
Perturbed material 42 (26054 from m400000): density=8.2751 g/cc
Perturbed material 32 (26054 from m300000): density=8.2751 g/cc


In [5]:
# Write all 4 perturbed materials to the MCNP input file
with open(workfile, "a") as f:
    f.write("c \n")
    f.write("c --- Perturbed materials (kika) ---\n")
    f.write(pert_m40_56.to_mcnp() + "\n")
    f.write(pert_m30_56.to_mcnp() + "\n")
    f.write(pert_m40_54.to_mcnp() + "\n")
    f.write(pert_m30_54.to_mcnp() + "\n")

print("Perturbed materials written:")
print(f"  m41 (26056 from m400000), m31 (26056 from m300000)")
print(f"  m42 (26054 from m400000), m32 (26054 from m300000)")

Perturbed materials written:
  m41 (26056 from m400000), m31 (26056 from m300000)
  m42 (26054 from m400000), m32 (26054 from m300000)


## 3. Generate PERT cards for both ZAIDs

In [6]:
# PERT cards for ZAID 26056 — reactions MT=2 and MT=51
pert_density_m40_56 = -pert_m40_56.metadata['density_info']['perturbed_mass_density']
pert_density_m30_56 = -pert_m30_56.metadata['density_info']['perturbed_mass_density']

kika.generate_PERTcards(
    inputfile=str(workfile),
    cell=[[3, 5, 7], [9]],
    reactions=[2, 51],
    material=[400000, 300000],
    energies=SCALE56,
    density=[-7.85, -7.85],
    order=1,
    nuclide=26056,
    in_place=True,
    pert_material=[41, 31],
    pert_density=[pert_density_m40_56, pert_density_m30_56],
)
# Expected: 2 materials x 2 reactions x 56 bins = 224 PERT cards (PERT1-224)


Success! PERT cards written to: test_output/PWRSphere_Fe56.i


In [7]:
# PERT cards for ZAID 26054 — reactions MT=2 and MT=51 (continues numbering from 225)
pert_density_m40_54 = -pert_m40_54.metadata['density_info']['perturbed_mass_density']
pert_density_m30_54 = -pert_m30_54.metadata['density_info']['perturbed_mass_density']

kika.generate_PERTcards(
    inputfile=str(workfile),
    cell=[[3, 5, 7], [9]],
    reactions=[2, 51],
    material=[400000, 300000],
    energies=SCALE56,
    density=[-7.85, -7.85],
    order=1,
    nuclide=26054,
    in_place=True,
    pert_material=[42, 32],
    pert_density=[pert_density_m40_54, pert_density_m30_54],
)
# Expected: another 224 PERT cards (PERT225-448)


Success! PERT cards written to: test_output/PWRSphere_Fe56.i


## 4. Verify: read back and inspect the perturbation cards

In [8]:
inp_pert = kika.read_mcnp(str(workfile))
print(inp_pert.perturbation)

# Verify totals
assert len(inp_pert.perturbation.pert) == 448, f"Expected 448, got {len(inp_pert.perturbation.pert)}"
print(f"\nTotal PERT cards: {len(inp_pert.perturbation.pert)}")
print(f"ZAIDs: {inp_pert.perturbation.zaids}")
print(f"Materials: {inp_pert.perturbation.materials}")
print(f"Reactions: {inp_pert.perturbation.reactions}")

assert inp_pert.perturbation.zaids == [26054, 26056], f"Expected [26054, 26056], got {inp_pert.perturbation.zaids}"
assert set(inp_pert.perturbation.materials) == {300000, 400000}
assert set(inp_pert.perturbation.reactions) == {2, 51}

# Verify sample PERT cards
pert1 = inp_pert.perturbation.pert[1]   # First card: 26056, m400000, MT=2
pert225 = inp_pert.perturbation.pert[225]  # First card: 26054, m400000, MT=2
print(f"\nPERT1:   ZAID={pert1.zaid}, MAT={pert1.material}, original_mat={pert1.original_material}, RXN={pert1.reaction}")
print(f"PERT225: ZAID={pert225.zaid}, MAT={pert225.material}, original_mat={pert225.original_material}, RXN={pert225.reaction}")

assert pert1.zaid == 26056 and pert1.material == 41 and pert1.original_material == 400000
assert pert225.zaid == 26054 and pert225.material == 42 and pert225.original_material == 400000

print("\nAll count and metadata assertions passed!")

                   MCNP Perturbation Data                   

Number of perturbations:  448
Perturbation numbers:     1-448
Particle types:           n
Reactions available:      2, 51
Methods available:        2
ZAIDs detected:           26054, 26056
Energy range:             1.00e-11 - 2.00e+01 MeV
Number of energy bins:    56
Energy structure:         SCALE56


Examples of accessing data:
- .pert[perturbation_number] - Access a specific perturbation


Total PERT cards: 448
ZAIDs: [26054, 26056]
Materials: [300000, 400000]
Reactions: [2, 51]

PERT1:   ZAID=26056, MAT=41, original_mat=400000, RXN=2
PERT225: ZAID=26054, MAT=42, original_mat=400000, RXN=2

All count and metadata assertions passed!


In [9]:
# Verify ZAID-aware grouping
for zaid, pert_mats in [(26056, {41, 31}), (26054, {42, 32})]:
    for mat in [400000, 300000]:
        for rxn in [2, 51]:
            group = inp_pert.perturbation._group_perts_by_reaction(2, material=mat, zaid=zaid)
            n = len(group.get(rxn, []))
            print(f"ZAID={zaid}, mat={mat}, MT={rxn}: {n} cards")
            assert n == 56, f"Expected 56, got {n}"

# Verify that without ZAID filter, both ZAIDs are mixed
g_all_m40 = inp_pert.perturbation._group_perts_by_reaction(2, material=400000)
print(f"\nWithout ZAID filter, m400000, MT=2: {len(g_all_m40.get(2, []))} cards (expect 112 = 2 ZAIDs x 56)")
assert len(g_all_m40.get(2, [])) == 112

# Verify materials_for_zaid
assert inp_pert.perturbation.materials_for_zaid(26056) == [300000, 400000]
assert inp_pert.perturbation.materials_for_zaid(26054) == [300000, 400000]

# Verify reactions_for_zaid
assert inp_pert.perturbation.reactions_for_zaid(26056) == [2, 51]
assert inp_pert.perturbation.reactions_for_zaid(26054) == [2, 51]

print("\nAll ZAID grouping assertions passed!")

ZAID=26056, mat=400000, MT=2: 56 cards
ZAID=26056, mat=400000, MT=51: 56 cards
ZAID=26056, mat=300000, MT=2: 56 cards
ZAID=26056, mat=300000, MT=51: 56 cards
ZAID=26054, mat=400000, MT=2: 56 cards
ZAID=26054, mat=400000, MT=51: 56 cards
ZAID=26054, mat=300000, MT=2: 56 cards
ZAID=26054, mat=300000, MT=51: 56 cards

Without ZAID filter, m400000, MT=2: 112 cards (expect 112 = 2 ZAIDs x 56)

All ZAID grouping assertions passed!


In [10]:
# DataFrame inspection
df = inp_pert.perturbation.to_dataframe()
print(f"Total PERT cards: {len(df)}")
print("\nBreakdown by ZAID, material, reaction:")
print(df.groupby(['zaid', 'original_material', 'reaction']).size().to_string())

Total PERT cards: 448

Breakdown by ZAID, material, reaction:
zaid   original_material  reaction
26054  300000             2           56
                          51          56
       400000             2           56
                          51          56
26056  300000             2           56
                          51          56
       400000             2           56
                          51          56


## 5. Inspect the generated MCNP input file (comment blocks)

In [11]:
# Verify per-block comments in the generated file
text = workfile.read_text()
lines = text.splitlines()

comment_lines = [l for l in lines if 'kika:pert_zaid=' in l]
print("Per-block comment lines:")
for cl in comment_lines:
    print(f"  {cl}")

# 4 comment blocks: 2 ZAIDs x 2 materials
assert len(comment_lines) == 4, f"Expected 4 per-block comments, got {len(comment_lines)}"
assert sum('pert_zaid=26056' in cl for cl in comment_lines) == 2
assert sum('pert_zaid=26054' in cl for cl in comment_lines) == 2

# Verify first PERT card of each ZAID has correct MAT
pert1_line = [l for l in lines if l.startswith('PERT1:n')][0]
pert225_line = [l for l in lines if l.startswith('PERT225:n')][0]
print(f"\nPERT1 (26056):   {pert1_line[:80]}...")
print(f"PERT225 (26054): {pert225_line[:80]}...")

assert 'MAT=41' in pert1_line, f"Expected MAT=41 in PERT1"
assert 'MAT=42' in pert225_line, f"Expected MAT=42 in PERT225"

print("\nFile content verification passed!")

Per-block comment lines:
  c kika:pert_zaid=26056 pert_mat=400000
  c kika:pert_zaid=26056 pert_mat=300000
  c kika:pert_zaid=26054 pert_mat=400000
  c kika:pert_zaid=26054 pert_mat=300000

PERT1 (26056):   PERT1:n CELL=3,5,7 MAT=41 RHO=-1.482784e+01 METHOD=2 RXN=2 ERG=1.000000e-11 4.00...
PERT225 (26054): PERT225:n CELL=3,5,7 MAT=42 RHO=-8.275139e+00 METHOD=2 RXN=2 ERG=1.000000e-11 4....

File content verification passed!
